# Fabric Insurance Demo — Prerequisite Deployment & Execution

**Purpose**: Single-notebook deployment and orchestration of all Fabric artifacts for the Insurance Demo.

**What this notebook does**:
1. Creates 3 Lakehouses (Bronze, Silver, Gold)
2. Creates 1 Warehouse
3. Discovers uploaded notebooks in the workspace
4. Executes all notebooks in order **with proper lakehouse bindings**
5. Creates cross-layer table shortcuts (Bronze→Silver, Silver→Gold) so notebooks can read from the previous layer

**Assumes**: The 13 pipeline notebooks have already been uploaded/imported to the workspace (manually or via Git integration).

**Mock data** is generated by `nb_00_generate_mock_data` (writes CSVs directly to Bronze Lakehouse).

**Prerequisites**:
- Run this notebook **inside your target Fabric workspace**
- Workspace must have Fabric capacity assigned
- User must have **Contributor** or **Admin** role on the workspace
- All 13 pipeline notebooks must already exist in the workspace

**Runtime**: ~20-30 minutes (includes all notebook execution)

## 0. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Update these values for your environment
# ============================================================

WORKSPACE_ID = "<YOUR_WORKSPACE_ID>"  # Find via: https://app.fabric.microsoft.com → Workspace URL

# Lakehouse names (per constitution naming convention)
LAKEHOUSES = [
    "lh_bronze_insurance",
    "lh_silver_insurance",
    "lh_gold_insurance"
]

# Warehouse name
WAREHOUSE_NAME = "wh_insurance"

# --- Notebook → Default Lakehouse mapping ---
# Bronze layer: reads CSVs from Files/, writes bronze tables
BRONZE_NOTEBOOKS = [
    "nb_00_generate_mock_data",
    "nb_01_bronze_ingest",
]

# Silver layer: reads bronze tables (via shortcuts), writes silver tables
SILVER_NOTEBOOKS = [
    "nb_02_silver_customers",
    "nb_03_silver_agents",
    "nb_04_silver_policies",
    "nb_05_silver_coverages",
    "nb_06_silver_premiums",
    "nb_07_silver_claims",
    "nb_08_silver_claim_payments",
]

# Gold layer: reads silver tables (via shortcuts), writes gold tables
GOLD_NOTEBOOKS = [
    "nb_09_gold_claims_summary",
    "nb_10_gold_premium_revenue",
    "nb_11_gold_customer_360",
    "nb_12_gold_kpi_metrics",
]

# Tables created in Bronze (nb_01) — shortcuts needed in Silver
BRONZE_TABLES = [
    "customers_raw", "agents_raw", "policies_raw",
    "coverages_raw", "premiums_raw", "claims_raw", "claim_payments_raw"
]

# Tables created in Silver (nb_02-nb_08) — shortcuts needed in Gold
SILVER_TABLES = [
    "customers", "agents", "policies",
    "coverages", "premiums", "claims", "claim_payments"
]

print("Configuration loaded")
print(f"   Workspace: {WORKSPACE_ID}")
print(f"   Lakehouses: {len(LAKEHOUSES)}")
print(f"   Bronze notebooks: {len(BRONZE_NOTEBOOKS)}")
print(f"   Silver notebooks: {len(SILVER_NOTEBOOKS)}")
print(f"   Gold notebooks:   {len(GOLD_NOTEBOOKS)}")

## 1. Authentication & Helper Functions

In [ ]:
import requests
import json
import time

# --- Authenticate using Fabric's built-in token provider ---
from notebookutils import mssparkutils

def get_fabric_token():
    """Get bearer token for Fabric REST API."""
    return mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

def get_auth_headers():
    return {
        "Authorization": f"Bearer {get_fabric_token()}",
        "Content-Type": "application/json"
    }

FABRIC_API = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}"

def fabric_post(endpoint, body, ignore_conflict=True):
    """POST to Fabric REST API with optional 409 conflict handling."""
    resp = requests.post(
        f"{FABRIC_API}/{endpoint}",
        headers=get_auth_headers(),
        json=body
    )
    if resp.status_code == 409 and ignore_conflict:
        return {"status": "already_exists", "code": 409}
    elif resp.status_code in [200, 201]:
        return resp.json() if resp.text else {"status": "created"}
    elif resp.status_code == 202:
        return {"status": "accepted", "location": resp.headers.get("Location", "")}
    else:
        print(f"   Error {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()

def fabric_get(endpoint):
    """GET from Fabric REST API."""
    resp = requests.get(
        f"{FABRIC_API}/{endpoint}",
        headers=get_auth_headers()
    )
    resp.raise_for_status()
    return resp.json()

# --- Run a notebook as a Fabric job with a specific default lakehouse ---
def run_notebook_job(notebook_id, notebook_name, lakehouse_name, lakehouse_id, timeout=900):
    """Run a notebook via Fabric REST API with a specific default lakehouse attached."""
    body = {
        "executionData": {
            "configuration": {
                "defaultLakehouse": {
                    "name": lakehouse_name,
                    "id": lakehouse_id,
                    "workspaceId": WORKSPACE_ID
                }
            }
        }
    }

    resp = requests.post(
        f"{FABRIC_API}/items/{notebook_id}/jobs/instances?jobType=RunNotebook",
        headers=get_auth_headers(),
        json=body
    )

    if resp.status_code not in [200, 202]:
        raise Exception(f"Failed to start {notebook_name}: {resp.status_code} - {resp.text[:300]}")

    # Get polling URL from Location header
    location = resp.headers.get("Location", "")

    if not location:
        # Fallback: wait and assume success if no polling URL
        print(f"    (no polling URL — waiting 60s)")
        time.sleep(60)
        return {"status": "assumed_complete"}

    # Poll for completion
    start = time.time()
    while time.time() - start < timeout:
        poll_resp = requests.get(location, headers=get_auth_headers())
        if poll_resp.status_code == 200:
            result = poll_resp.json()
            status = result.get("status", "Unknown")
            if status in ["Completed", "Succeeded"]:
                return result
            elif status in ["Failed", "Cancelled"]:
                error_detail = json.dumps(result, indent=2)[:500]
                raise Exception(f"{notebook_name} {status}: {error_detail}")
        time.sleep(15)

    raise TimeoutError(f"{notebook_name} timed out after {timeout}s")

# --- Create a OneLake table shortcut (so notebooks can read cross-lakehouse) ---
def create_table_shortcut(target_lh_id, source_lh_id, table_name):
    """Create a table shortcut in target lakehouse pointing to a source lakehouse table."""
    body = {
        "name": table_name,
        "path": "Tables",
        "target": {
            "oneLake": {
                "itemId": source_lh_id,
                "path": f"Tables/{table_name}",
                "workspaceId": WORKSPACE_ID
            }
        }
    }

    resp = requests.post(
        f"{FABRIC_API}/items/{target_lh_id}/shortcuts",
        headers=get_auth_headers(),
        json=body
    )

    if resp.status_code in [200, 201]:
        return "created"
    elif resp.status_code == 409:
        return "exists"
    else:
        print(f"    Warning: shortcut '{table_name}' failed ({resp.status_code}): {resp.text[:200]}")
        return "failed"

# --- Validate connection ---
try:
    ws_info = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}",
        headers=get_auth_headers()
    ).json()
    print(f"Connected to workspace: {ws_info.get('displayName', 'Unknown')}")
    print(f"   Workspace ID: {WORKSPACE_ID}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("   Make sure WORKSPACE_ID is correct and you have Contributor access.")

## 2. Create Lakehouses (Bronze, Silver, Gold)

In [ ]:
print("=" * 60)
print("STEP 2: Creating Lakehouses")
print("=" * 60)

lakehouse_ids = {}

for lh_name in LAKEHOUSES:
    print(f"\n  Creating {lh_name}...")
    result = fabric_post("items", {
        "displayName": lh_name,
        "type": "Lakehouse"
    })

    if result.get("status") == "already_exists":
        print(f"    Skipped — {lh_name} already exists")
    else:
        print(f"    Created {lh_name}")

# Brief pause for propagation, then retrieve IDs
time.sleep(3)
items = fabric_get("items?type=Lakehouse")
for item in items.get("value", []):
    if item["displayName"] in LAKEHOUSES:
        lakehouse_ids[item["displayName"]] = item["id"]
        print(f"    {item['displayName']} -> {item['id']}")

print(f"\nLakehouses ready: {len(lakehouse_ids)}/{len(LAKEHOUSES)}")

## 3. Create Warehouse

In [ ]:
print("=" * 60)
print("STEP 3: Creating Warehouse")
print("=" * 60)

print(f"\n  Creating {WAREHOUSE_NAME}...")
result = fabric_post("items", {
    "displayName": WAREHOUSE_NAME,
    "type": "Warehouse"
})

if result.get("status") == "already_exists":
    print(f"    Skipped — {WAREHOUSE_NAME} already exists")
else:
    print(f"    Created {WAREHOUSE_NAME}")

# Retrieve warehouse ID
time.sleep(3)
warehouse_id = None
items = fabric_get("items?type=Warehouse")
for item in items.get("value", []):
    if item["displayName"] == WAREHOUSE_NAME:
        warehouse_id = item["id"]
        print(f"    {WAREHOUSE_NAME} -> {warehouse_id}")
        break

print(f"\nWarehouse ready")

## 4. Discover Notebooks in Workspace

In [ ]:
print("=" * 60)
print("STEP 4: Discovering Notebooks in Workspace")
print("=" * 60)

ALL_NOTEBOOKS = BRONZE_NOTEBOOKS + SILVER_NOTEBOOKS + GOLD_NOTEBOOKS
notebook_ids = {}

items = fabric_get("items?type=Notebook")
for item in items.get("value", []):
    if item["displayName"] in ALL_NOTEBOOKS:
        notebook_ids[item["displayName"]] = item["id"]

print(f"\n  Found {len(notebook_ids)}/{len(ALL_NOTEBOOKS)} notebooks:\n")
for nb_name in ALL_NOTEBOOKS:
    found = nb_name in notebook_ids
    icon = "OK" if found else "MISSING"
    nb_id = notebook_ids.get(nb_name, "---")
    short_id = nb_id[:12] + "..." if len(nb_id) > 12 else nb_id
    print(f"    [{icon:>7s}] {nb_name:<40} {short_id}")

missing = [nb for nb in ALL_NOTEBOOKS if nb not in notebook_ids]
if missing:
    print(f"\n  WARNING: {len(missing)} notebooks not found. Upload them before continuing:")
    for nb in missing:
        print(f"    - {nb}")
    raise Exception(f"Missing notebooks: {missing}")
else:
    print(f"\n  All {len(ALL_NOTEBOOKS)} notebooks found. Ready for execution.")

## 5. Execute Bronze Layer

Runs `nb_00` (mock data generation) and `nb_01` (CSV → Delta ingest) with **lh_bronze_insurance** as the default lakehouse.

After Bronze completes, creates **table shortcuts** in `lh_silver_insurance` pointing to all Bronze `*_raw` tables so Silver notebooks can read them.

In [ ]:
print("=" * 60)
print("STEP 5: Execute Bronze Layer")
print("=" * 60)

bronze_lh_id = lakehouse_ids["lh_bronze_insurance"]
silver_lh_id = lakehouse_ids["lh_silver_insurance"]

print(f"\n  Default Lakehouse: lh_bronze_insurance ({bronze_lh_id[:12]}...)")
print()

bronze_results = []
for nb_name in BRONZE_NOTEBOOKS:
    nb_id = notebook_ids[nb_name]
    print(f"  Running {nb_name}...")
    start_time = time.time()

    try:
        run_notebook_job(nb_id, nb_name, "lh_bronze_insurance", bronze_lh_id)
        elapsed = time.time() - start_time
        print(f"    Completed ({elapsed:.0f}s)")
        bronze_results.append((nb_name, "SUCCESS", elapsed))
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"    FAILED ({elapsed:.0f}s): {str(e)[:300]}")
        bronze_results.append((nb_name, "FAILED", elapsed))
        raise Exception(f"Bronze pipeline failed at {nb_name}. Fix the notebook and re-run this cell.")

# --- Create table shortcuts: Bronze → Silver ---
print(f"\n  Creating table shortcuts: lh_bronze_insurance -> lh_silver_insurance...")
shortcut_results = []
for table in BRONZE_TABLES:
    status = create_table_shortcut(silver_lh_id, bronze_lh_id, table)
    shortcut_results.append(status)
    print(f"    {table}: {status}")

created = sum(1 for s in shortcut_results if s in ["created", "exists"])
print(f"\n  Bronze layer complete. {created}/{len(BRONZE_TABLES)} shortcuts ready in Silver lakehouse.")

## 6. Execute Silver Layer

Runs `nb_02` through `nb_08` with **lh_silver_insurance** as the default lakehouse. Each notebook reads Bronze `*_raw` tables via shortcuts and writes cleansed tables to Silver.

After Silver completes, creates **table shortcuts** in `lh_gold_insurance` pointing to all Silver tables so Gold notebooks can read them.

In [ ]:
print("=" * 60)
print("STEP 6: Execute Silver Layer")
print("=" * 60)

silver_lh_id = lakehouse_ids["lh_silver_insurance"]
gold_lh_id = lakehouse_ids["lh_gold_insurance"]

print(f"\n  Default Lakehouse: lh_silver_insurance ({silver_lh_id[:12]}...)")
print(f"  Reading bronze data via table shortcuts.")
print()

silver_results = []
for nb_name in SILVER_NOTEBOOKS:
    nb_id = notebook_ids[nb_name]
    print(f"  Running {nb_name}...")
    start_time = time.time()

    try:
        run_notebook_job(nb_id, nb_name, "lh_silver_insurance", silver_lh_id)
        elapsed = time.time() - start_time
        print(f"    Completed ({elapsed:.0f}s)")
        silver_results.append((nb_name, "SUCCESS", elapsed))
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"    FAILED ({elapsed:.0f}s): {str(e)[:300]}")
        silver_results.append((nb_name, "FAILED", elapsed))
        raise Exception(f"Silver pipeline failed at {nb_name}. Fix the notebook and re-run this cell.")

# --- Create table shortcuts: Silver → Gold ---
print(f"\n  Creating table shortcuts: lh_silver_insurance -> lh_gold_insurance...")
shortcut_results = []
for table in SILVER_TABLES:
    status = create_table_shortcut(gold_lh_id, silver_lh_id, table)
    shortcut_results.append(status)
    print(f"    {table}: {status}")

created = sum(1 for s in shortcut_results if s in ["created", "exists"])
print(f"\n  Silver layer complete. {created}/{len(SILVER_TABLES)} shortcuts ready in Gold lakehouse.")

## 7. Execute Gold Layer

Runs `nb_09` through `nb_12` with **lh_gold_insurance** as the default lakehouse. Each notebook reads Silver tables via shortcuts and writes business aggregates to Gold.

In [ ]:
print("=" * 60)
print("STEP 7: Execute Gold Layer")
print("=" * 60)

gold_lh_id = lakehouse_ids["lh_gold_insurance"]

print(f"\n  Default Lakehouse: lh_gold_insurance ({gold_lh_id[:12]}...)")
print(f"  Reading silver data via table shortcuts.")
print()

gold_results = []
for nb_name in GOLD_NOTEBOOKS:
    nb_id = notebook_ids[nb_name]
    print(f"  Running {nb_name}...")
    start_time = time.time()

    try:
        run_notebook_job(nb_id, nb_name, "lh_gold_insurance", gold_lh_id)
        elapsed = time.time() - start_time
        print(f"    Completed ({elapsed:.0f}s)")
        gold_results.append((nb_name, "SUCCESS", elapsed))
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"    FAILED ({elapsed:.0f}s): {str(e)[:300]}")
        gold_results.append((nb_name, "FAILED", elapsed))
        raise Exception(f"Gold pipeline failed at {nb_name}. Fix the notebook and re-run this cell.")

print(f"\n  Gold layer complete.")

## 8. Pipeline Summary

In [ ]:
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)

all_results = bronze_results + silver_results + gold_results
total_time = sum(e for _, _, e in all_results)

print(f"\n  EXECUTION SUMMARY")
print(f"  " + "-" * 56)
print(f"  {'Notebook':<42} {'Status':>8} {'Time':>6}")
print(f"  " + "-" * 56)

for nb_name, status, elapsed in all_results:
    icon = "OK" if status == "SUCCESS" else "FAIL"
    layer = "Bronze" if ("bronze" in nb_name or "mock" in nb_name) else "Silver" if "silver" in nb_name else "Gold"
    print(f"  [{icon:>4s}] [{layer:6s}] {nb_name:<34} {elapsed:>5.0f}s")

success_count = sum(1 for _, s, _ in all_results if s == "SUCCESS")
print(f"\n  Succeeded: {success_count}/{len(all_results)}")
print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")

print(f"""
  -----------------------------------------------
  Artifacts & Data Flow:
  -----------------------------------------------
  Lakehouses:
    lh_bronze_insurance  -> {len(BRONZE_TABLES)} raw tables
    lh_silver_insurance  -> {len(SILVER_TABLES)} cleansed tables (+ shortcuts to bronze)
    lh_gold_insurance    -> 4 aggregate tables (+ shortcuts to silver)

  Warehouse: {WAREHOUSE_NAME}

  -----------------------------------------------
  Next Steps:
  -----------------------------------------------
  1. Create Warehouse shortcuts to Gold Lakehouse tables
  2. Run warehouse/sample_queries.sql against {WAREHOUSE_NAME}
  3. Create Power BI semantic model (sm_insurance)
     using DirectLake mode against lh_gold_insurance tables:
     - gold_claims_summary
     - gold_premium_revenue
     - gold_customer_360
     - gold_kpi_metrics
  4. Build Power BI report (rpt_insurance_dashboard)

  -----------------------------------------------
  Workspace URL:
  https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}
""")